---
title: Handling Categorical Feature
description: A comprehensive guide to transforming non-numeric categories into structured,
  numeric formats that algorithms can effectively process and learn from.
author: Md Muztahid Hassan
date: '2026-05-20 21:00:00'
toc: true
execute:
  warning: false
  echo: true
order: 12
---


## Introduction

Categorical features are variables that contain text labels or categories rather than numerical values (for example: "Color," "City," or "Size").

The core challenge with categorical data is that most machine learning algorithms are rooted in mathematics. They require numerical inputs to calculate equations, distances, and probabilities. If you try to feed raw text like "New York" or "Red" into a model, it will throw an error. Therefore, handling categorical features is the process of translating these text categories into a numerical format that an algorithm can understand, without losing the underlying information.

### The Two Main Techniques

There are two primary ways to convert categorical data into numbers, depending on whether the data has a natural order:

**1. Ordinal Encoding (For Ordered Categories)**
Use this when your categories have a clear, logical ranking or hierarchy. The technique assigns a distinct integer to each category based on its order.

* **Example:** A "Size" column with values `Small`, `Medium`, `Large`.
* **Transformation:** `Small` becomes **1**, `Medium` becomes **2**, and `Large` becomes **3**. The model understands that 3 is greater than 1, preserving the inherent rank.

**2. One-Hot Encoding (For Unordered/Nominal Categories)**
Use this when your categories do not have any mathematical relationship or ranking to one another. Instead of assigning a single number (which might trick the model into thinking one category is "greater" than another), One-Hot Encoding creates a new, separate column for every unique category. Each column is filled with **1** (if the category is present) or **0** (if it is not).

* **Example:** A "Color" column with values `Red`, `Green`, `Blue`.
* **Transformation:** It splits into three new columns: `Is_Red`, `Is_Green`, and `Is_Blue`. A red item would have a **1** in the `Is_Red` column and a **0** in the others.

That means if there are ranks in categorical data, we use `ordinal encoding`. Otherwise, we use `one hot encoding` for nominal data.


### When to Use Label Encoding: The Target Variable

The most common—and mathematically correct—use case for Label Encoding is transforming the **target variable** (the specific outcome your model is trying to predict) into a machine-readable format.

While applying Label Encoding to input features can sometimes confuse distance-based algorithms into thinking categories have a numerical rank (e.g., assuming category `3` is "greater" than category `1`), the target variable does not have this problem. The target is simply the final classification output. In fact, popular machine learning libraries like `scikit-learn` explicitly expect the target array (typically denoted as `y`) to be formatted as a single column of encoded integers.

**A Quick Example:**
Imagine you are building a classification model to predict whether an email is spam.

* **Your Raw Target Data:** `["Spam", "Not Spam", "Not Spam", "Spam"]`
* **Your Encoded Target Data:** `[1, 0, 0, 1]`

By restricting Label Encoding strictly to your target variable, you ensure your algorithm can efficiently process and predict the correct classes without accidentally skewing the mathematical relationships within your input features.


Here is a comparison table designed to be clear and reader-friendly for your blog. It breaks down the key distinctions between the three encoding methods:

| Feature | Ordinal Encoding | Nominal (One-Hot) Encoding | Label Encoding |
| --- | --- | --- | --- |
| **Best Used For** | Categorical **input features** with a natural order. | Categorical **input features** with no natural order. | The **target variable** (the outcome you are trying to predict). |
| **How It Works** | Assigns integers based on a logical, predefined hierarchy. | Creates a new binary column (0 or 1) for every unique category. | Assigns unique integers to categories arbitrarily (usually alphabetically). |
| **Mathematical Assumption** | The numbers have a meaningful rank (e.g., 3 is greater than 1). | All categories are strictly equal in weight and distance. | The numbers imply a rank, which can confuse some algorithms if used on input features. |
| **Dimensionality** | Keeps the dataset size the same (1 column remains 1 column). | Expands the dataset (1 column becomes N columns). | Keeps the dataset size the same (1 column remains 1 column). |
| **Algorithm Suitability** | Works well with both linear/distance-based and tree-based models. | Essential for linear/distance-based models (Linear Regression, KNN). | Safe for tree-based models (Random Forest, XGBoost) and target arrays. |
| **Example Transformation** | **Small, Medium, Large** becomes **1, 2, 3** | **Red, Blue** becomes **Is_Red, Is_Blue** | **Not Spam, Spam** becomes **0, 1** |


> Note: When we will work with input features, we will use ordinal encoding or nominal encoding according to our need. But when we will see that the target variable has categorical value, we can use label encoding.

## Ordinal Encoding
If a question arises that how computer knows the rank by ordinal encoding? We can explain that with an example. Suppose the `Education` feature has two value: `MSc` and `BSc`. The rank of `MSc` is greater that the rank of `BSc`. We assign greater value to the `MSc`(suppose 2) and relatively smaller value to `BSc`(suppose 1). This is how the computer recognizes the rank.

Now let's see them in practical.
First, import libraries and load the dataset. You can download the dataset from <a href="customer.csv" download>here</a>.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer.csv")
df.head()

,age,gender,review,education,purchased
0,30,Female,Average,School,No
1,68,Female,Poor,UG,No
2,70,Female,Good,PG,No
3,72,Female,Good,PG,No
4,16,Female,Average,UG,No


Here,
`age` -> Numeric Feature
`gender` -> Nominal
`review` -> Ordinal
`education` -> Ordinal
`purchased` -> Label Encoding (Because this is the target variable)
Actually we won't use the `age`and `gender` columns for our learning. So, we will omit them.

In [2]:
df = df.iloc[:,2:]
df.sample(5)

,review,education,purchased
34,Average,School,No
15,Poor,UG,No
19,Poor,PG,Yes
47,Good,PG,Yes
37,Average,PG,Yes


In [3]:
df.shape

(50, 3)

In [4]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(df.drop('purchased', axis=1), df['purchased'], test_size=0.3, random_state=0)

x_train.shape, x_test.shape

((35, 2), (15, 2))

During fitting the data into ordinal encoding, we need to do some work manually. We need to specify the ranks manually from lower to higher. Otherwise, it will be randomly ranked.

In [5]:
oe = OrdinalEncoder(categories=[['Poor','Average','Good'],['School','UG','PG']])
oe.fit(x_train)

x_train = pd.DataFrame(oe.transform(x_train), columns=x_train.columns)
x_test = pd.DataFrame(oe.transform(x_test), columns=x_test.columns)
x_train.head()

,review,education
0,0.0,0.0
1,0.0,2.0
2,0.0,2.0
3,2.0,1.0
4,1.0,1.0


In [6]:
x_test.head()

,review,education
0,0.0,0.0
1,2.0,1.0
2,2.0,1.0
3,2.0,2.0
4,2.0,2.0


In [7]:
oe.categories_

[array(['Poor', 'Average', 'Good'], dtype=object),
 array(['School', 'UG', 'PG'], dtype=object)]

Now we will use label encoder on our target column `purchased`.

In [8]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
le.fit(y_train)

LabelEncoder()

In [9]:
le.classes_

array(['No', 'Yes'], dtype=object)

In [10]:
y_train = le.transform(y_train)
y_test = le.transform(y_test)

In [11]:
y_train

array([1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0])

## One-Hot Encoding

Suppose we have a table with feature `fuel`.

|fuel|
|---|
|Diesel|
|Petrol|
|Octen|
|Diesel|
|Octen|

When we will do one-hot encoding on the fuel feature, the encoder will try to find out the number of unique values present in the feature. Then it will create a table with each unique value as a column.

Because there are three unique categories in your raw data (**Diesel**, **Petrol**, and **Octen**), One-Hot Encoding splits them into three separate binary columns.

| Diesel | Petrol | Octen |
| --- | --- | --- |
| 1 | 0 | 0 |
| 0 | 1 | 0 |
| 0 | 0 | 1 |
| 1 | 0 | 0 |
| 0 | 0 | 1 |

**How it works:**
For each row, the system places a **1** in the column that matches the original category and a **0** in all the other columns. This perfectly translates the text into a mathematical format without accidentally tricking the algorithm into thinking one fuel type is "greater" or "less" than another.

We can create the table by omiting any one of the columns. Suppose we omit `Diesel` column. Then if we find all values of other columns are **0**, then it is indicating the `Diesel` value. This is necessary when we will work with `Linear Models`. Otherwise, it is optional.

### The Drawbacks of One-Hot Encoding

While it is the standard for nominal data, One-Hot Encoding has four major pitfalls:

* **The Curse of Dimensionality:** It creates a massive number of new columns for features with many unique categories, making it harder for models to find patterns.
* **Extreme Sparsity:** It generates a dataset filled almost entirely with zeroes, which consumes excessive memory and slows down training times.
* **The Dummy Variable Trap:** It introduces perfect multicollinearity (features predicting each other), which will break linear models unless you drop one of the encoded columns.
* **Production Brittleness:** It will crash or throw errors in a real-world environment if it encounters a brand-new category it did not see during training.

Now let's see them in practical.

In [12]:
df = pd.read_csv("customer.csv")
df.head()

,age,gender,review,education,purchased
0,30,Female,Average,School,No
1,68,Female,Poor,UG,No
2,70,Female,Good,PG,No
3,72,Female,Good,PG,No
4,16,Female,Average,UG,No


We will use one-hot encoding on the `gender` column as it is a nominal feature. It we do so, we will find two columns because we have two unique values. Now the number of total columns is 5 and after applying one-hot encoding, there will be 7 columns.

In [13]:
df['gender'].value_counts()

gender
Female    29
Male      21
Name: count, dtype: int64

In [14]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)
new_data = ohe.fit_transform(df[['gender']])
pd.DataFrame(new_data, columns=["Female", "Male"]).head()

,Female,Male
0,1.0,0.0
1,1.0,0.0
2,1.0,0.0
3,1.0,0.0
4,1.0,0.0


Look here we can see that it has created two columns for two unique values.

We have one problem that is, we have to add these two columns to the main dataset and remove the `gender` column. This manual process is time consuming. We can use `get_dummies` function to automatically add them.

In [15]:
pd.get_dummies(df, columns=['gender']).head()

,age,review,education,purchased,gender_Female,gender_Male
0,30,Average,School,No,True,False
1,68,Poor,UG,No,True,False
2,70,Good,PG,No,True,False
3,72,Good,PG,No,True,False
4,16,Average,UG,No,True,False


Here **True** means **1** and **False** means **0**. There are also some limitations of this function.

* **Production Brittleness:** It doesn't "remember" your training categories. If new data has unseen or missing categories, it generates the wrong number of columns and crashes your model.
* **Pipeline Incompatibility:** It is a Pandas function, not a Scikit-Learn transformer, meaning you cannot cleanly integrate it into automated machine learning pipelines.

> Note: Use `get_dummies()` for quick analysis, but always use `OneHotEncoder` for real models!.

So, we will use `Pipelines` or `Column Transformation` in future.